# Stage 2 Notebook 15 - Exp2J CLRKD + separate cls feature pathway

Exp2I (NB14) produced the strongest cls signal yet: at epoch 2 (first epoch of `adapter_warmup`) `val/lane_exist_best_f1 = 0.5355` with `pos_score - neg_score = +0.163` -- *above* the project's pass target on separation. Then the signal collapsed to 0.057 by epoch 10 once `full_finetune` opened the full backbone.

Phase correlation is decisive. The cls signal peaks the moment GCA + adapters unfreeze, and it dies the moment full-backbone gradients flow. On training (not just val), `train/lane/cls_pos` rose 0.083 -> 0.13 over the run mirroring the val regression -- the model is genuinely losing cls capability, not just generalizing poorly.

Diagnosis: the cls head reads `per_lane`, the same 128-d feature consumed by `param_head` and `offset_head`. Geometry losses dominate the gradient flow into `per_lane` -- they reshape it to encode 'what curve goes through here' which is *similar* across geometrically-similar priors near the same GT lane. The cls task ('is *this specific* prior matched?') needs the opposite: a feature that distinguishes the dynamic-k winner from its similar neighbors. OHEM + ASL + prior encoder cannot recover this once the feature is curve-specialized, because the bottleneck is gradient flow into shared features, not loss formulation.

Exp2J builds a **parallel cls aggregator**:

- Same per-scale ROI samples (sampled along each prior's curve) feed both branches -- the bilinear sample positions still see gradient from both losses through curve geometry.
- Geometry branch: existing `scale_blocks`, `scale_fusion`, `fc`, `cross_attn` -> `per_lane_geom` -> `param_head`, `offset_head`.
- New cls branch: parallel `scale_blocks_cls`, `scale_fusion_cls`, `fc_cls`, `cross_attn_cls` -> `per_lane_cls` -> `cls_head` (with prior_embed_encoder concat).
- Each branch's parameters get only its own loss's gradient. Cls features are no longer corrupted by geometry-driven feature reshaping.

Hypothesis: if the epoch-2 spike in Exp2I is the cls head's natural performance with stable features, decoupling cls from geometry-driven feature drift should let that signal *persist* through full_finetune.

Other tweaks: `w_iou: 1.5 -> 2.0` (recover Exp2G's geometry weight; with cls now isolated we don't need to soften geometry). All other settings (OHEM topk_per_pos=4 + min_topk=32, prior_embed_encoder_dim=64, ASL gamma_pos=0/gamma_neg=4, dynamic-k, 3 stages, RMT+GCA, lambda_min=0.5) identical to Exp2I.

Cost: forward time and memory roughly +50% on the lane head only (post-sample aggregator doubled). The shared backbone + grid_sample dominate compute, so end-to-end overhead should be under 15%.

Reference: external_repos/CLRNet/clrnet/models/utils/roi_gather.py (single aggregator design that this experiment doubles).

### Run mode

1. Keep `DEBUG_MODE = True` for the first run.
2. After the smoke and debug run succeed, change to `False` for the 10-epoch short run.
3. Output is mirrored to the notebook cell, the Colab runtime log, and a Drive log file.
4. Do not rerun Notebook 00.

In [2]:
import os, sys, subprocess, textwrap
from google.colab import drive
os.environ['PYTHONUNBUFFERED'] = '1'
drive.mount('/content/drive')

REPO_ROOT = '/content/drive/MyDrive/EcoCAR/yolop_vehicle_lane'
if not os.path.isdir(REPO_ROOT):
    raise FileNotFoundError(f'Missing project root: {REPO_ROOT}')
os.chdir(REPO_ROOT)
if REPO_ROOT not in sys.path:
    sys.path.insert(0, REPO_ROOT)

subprocess.check_call([sys.executable, '-m', 'pip', 'install', '-q', 'pyyaml', 'scipy', 'opencv-python-headless', 'tqdm', 'matplotlib'])
print('repo:', REPO_ROOT)

from stage2.scripts.notebook_utils import run_streaming
LOG_DIR = '/content/drive/MyDrive/EcoCAR/training_runs/notebook_logs'
os.makedirs(LOG_DIR, exist_ok=True)

Mounted at /content/drive
repo: /content/drive/MyDrive/EcoCAR/yolop_vehicle_lane


In [3]:
from pathlib import Path
import os, sys

CONFIG = 'stage2/configs/exp10_rmt_gca_clrkd_separate_cls_path_joint.yaml'
LOG_FILE = os.path.join(LOG_DIR, f'{Path(CONFIG).stem}_smoke.log')

# Smoke test: forward + backward through doubled aggregator (geom + cls).
# Must print 'OK exp10_*.yaml' with shapes lane_shape=(1, 16, 72, 2)
# det_shape=(1, 4, 4) before training is attempted.
run_streaming([sys.executable, '-u', 'stage2/scripts/smoke_test_joint_models.py', CONFIG], log_path=LOG_FILE)

[run_streaming] command: /usr/bin/python3 -u stage2/scripts/smoke_test_joint_models.py stage2/configs/exp10_rmt_gca_clrkd_separate_cls_path_joint.yaml
[run_streaming] log file: /content/drive/MyDrive/EcoCAR/training_runs/notebook_logs/exp10_rmt_gca_clrkd_separate_cls_path_joint_smoke.log
OK exp10_rmt_gca_clrkd_separate_cls_path_joint.yaml
  lane_shape=(1, 16, 72, 2) det_shape=(1, 4, 4)
  lane_loss=4.3092 det_loss=3.1773 grad_cos=-0.1389 lambda_lane=0.0500
  gate_stats={'gate/det_mean': 0.501537561416626, 'gate/lane_mean': 0.4995344579219818, 'gate/det_sat_low': 0.0, 'gate/det_sat_high': 0.0, 'gate/lane_sat_low': 0.0, 'gate/lane_sat_high': 0.0}
[run_streaming] return_code=0


0

In [4]:
from pathlib import Path
import os, sys

CONFIG = 'stage2/configs/exp10_rmt_gca_clrkd_separate_cls_path_joint.yaml'
CURVE_TAR = '/content/drive/MyDrive/EcoCAR/datasets/bdd100k_clrkd_curve.tar'
CURVE_ROOT = '/content/bdd100k_clrkd_curve'

DEBUG_MODE = False

if DEBUG_MODE:
    RUN_TAG = 'debug'
    EPOCHS = 2
    BATCH_SIZE = 4
    LIMIT_TRAIN = 512
    LIMIT_VAL = 256
    PRINT_EVERY = 5
else:
    RUN_TAG = 'short10'
    EPOCHS = 10
    BATCH_SIZE = 8
    LIMIT_TRAIN = 3000
    LIMIT_VAL = 1000
    PRINT_EVERY = 5

run_stem = Path(CONFIG).stem + '_' + RUN_TAG
WORK_DIR = f'/content/{run_stem}'
OUTPUT_TAR = f'/content/drive/MyDrive/EcoCAR/training_runs/{run_stem}.tar'
LOG_FILE = os.path.join(LOG_DIR, f'{run_stem}_train.log')

cmd = [
    sys.executable, '-u', 'stage2/scripts/train_joint_model_experiment.py',
    '--config', CONFIG,
    '--curve-tar', CURVE_TAR,
    '--curve-root', CURVE_ROOT,
    '--work-dir', WORK_DIR,
    '--output-tar', OUTPUT_TAR,
    '--epochs', str(EPOCHS),
    '--batch-size', str(BATCH_SIZE),
    '--limit-train', str(LIMIT_TRAIN),
    '--limit-val', str(LIMIT_VAL),
    '--force-extract',
    '--print-every', str(PRINT_EVERY),
]

print('DEBUG_MODE:', DEBUG_MODE, flush=True)
print('About to run:', ' '.join(cmd), flush=True)
print('Output tar:', OUTPUT_TAR, flush=True)
print('Visible log file:', LOG_FILE, flush=True)
run_streaming(cmd, log_path=LOG_FILE)

DEBUG_MODE: False
About to run: /usr/bin/python3 -u stage2/scripts/train_joint_model_experiment.py --config stage2/configs/exp10_rmt_gca_clrkd_separate_cls_path_joint.yaml --curve-tar /content/drive/MyDrive/EcoCAR/datasets/bdd100k_clrkd_curve.tar --curve-root /content/bdd100k_clrkd_curve --work-dir /content/exp10_rmt_gca_clrkd_separate_cls_path_joint_short10 --output-tar /content/drive/MyDrive/EcoCAR/training_runs/exp10_rmt_gca_clrkd_separate_cls_path_joint_short10.tar --epochs 10 --batch-size 8 --limit-train 3000 --limit-val 1000 --force-extract --print-every 5
Output tar: /content/drive/MyDrive/EcoCAR/training_runs/exp10_rmt_gca_clrkd_separate_cls_path_joint_short10.tar
Visible log file: /content/drive/MyDrive/EcoCAR/training_runs/notebook_logs/exp10_rmt_gca_clrkd_separate_cls_path_joint_short10_train.log
[run_streaming] command: /usr/bin/python3 -u stage2/scripts/train_joint_model_experiment.py --config stage2/configs/exp10_rmt_gca_clrkd_separate_cls_path_joint.yaml --curve-tar /con

0

## What to watch in Exp2J training

Reference epoch 10 across recent experiments:
- Exp2G: point_mae=0.3244, matched_iou=0.4285, best_f1=0.075, pos-neg=0.004.
- Exp2H: point_mae=0.3285, matched_iou=0.4018, best_f1=0.083, pos-neg=0.016.
- Exp2I: point_mae=0.3293, matched_iou=0.3911, best_f1=0.057, pos-neg=0.001 (peaked at e2: best_f1=0.5355, pos-neg=+0.163, then collapsed).

**The persistence test** (the entire point of Exp2J): does the Exp2I-style epoch-2 cls signal *persist* across full_finetune?

Strong signals that the separate cls pathway worked:

- `val/lane_exist_best_f1` >= 0.40 by epoch 5 and >= 0.65 by epoch 10. Most importantly: it does NOT collapse from epoch 3 onward. (Exp2I went 0.535 -> 0.219 -> 0.115 -> 0.071 -> ... -> 0.057.)
- `val/lane_exist_pos_score_mean - val/lane_exist_neg_score_mean` >= 0.15 at epoch 10 with no decay. (Exp2I went +0.163 -> +0.001.)
- `val/lane/cls_pos` strictly DECREASES over training. With cls features no longer being reshaped by geometry, positive predictions should improve, not regress.
- `pred_lanes / batch` drops below 500 (Exp2G/H/I were stuck at ~1500 = all 192*8 priors above thr=0.3).
- **Geometry holds or improves**: `val/lane_point_mae <= 0.34` and `val/matched_line_iou >= 0.40` at epoch 10. With w_iou raised back to 2.0 we expect matched_iou to match or exceed Exp2G's 0.428.

Failure signals -> next ablation:

- best_f1 still below 0.20 at epoch 10 -> the bilinear sample positions themselves are the bottleneck (cls aggregator's input ROI sample locations are pulled by geometry through curve params). Next: **stop-gradient from cls into curve params** so the cls aggregator reads features at geometry-driven positions only. Or replace cls with thresholded-LineIoU-regression.
- Geometry regresses (point_mae > 0.36 or matched_iou < 0.30) -> the cls aggregator is stealing capacity from the backbone via its grad path. Reduce parameter sharing: keep `scale_blocks` shared (early per-scale conv) and only diverge at `scale_fusion + cross_attn`.
- Forward pass cost or memory increases >= 30% vs Exp2I -> reduce `roi_mid_channels` from 48 to 32 in the cls path only.

After short10 finishes, run Notebook 08 (now includes exp10 candidates) to plot Exp2G / Exp2H / Exp2I / Exp2J side-by-side.